# Medical ICD-10 Code Classification
**Non-deep-learning NLP pipeline for classifying Spanish/Catalan medical literals to ICD-10 codes**

## Pipeline Overview
1. **Regex shortcut** -- if the literal is already a valid ICD code, return it directly
2. **Preprocessing** -- lowercase, remove accents, expand abbreviations, translate Catalan terms
3. **Primary classifier** -- Word + character n-gram TF-IDF + Logistic Regression
4. **Fallback** -- cosine similarity against ICD-10 official descriptions

In [1]:
# !pip install pandas scikit-learn numpy scipy tqdm -q

In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Data

In [3]:
train_df = pd.read_csv('data/raw/training_codification_data.csv')
test_df  = pd.read_csv('data/raw/test_leaderboard_data.csv')
icd_df   = pd.read_csv('data/raw/icd_d_p_pairs.csv')

print(f'Training samples : {len(train_df):,}  |  Unique codes: {train_df["Code"].nunique():,}')
print(f'Test samples     : {len(test_df):,}')
print(f'ICD dictionary   : {len(icd_df):,} entries')
icd_codes_set = set(icd_df['Code'].astype(str))
coverage = train_df['Code'].astype(str).isin(icd_codes_set).mean()
print(f'Training codes in ICD dict: {coverage:.1%}')
train_df.head()

Training samples : 13,700  |  Unique codes: 4,059
Test samples     : 6,667
ICD dictionary   : 179,742 entries
Training codes in ICD dict: 72.6%


,Code,Literal
0,J9809,Hiperreactividad bronquial
1,J9801,broncoespástica
2,I420,miocardiopatía dilatada
3,Y831,HTA irc 6
4,R5600,Crisis febriles atípicas


## 2. Domain Dictionaries

In [4]:
# Medical Spanish abbreviation expansion
ABBREVIATIONS = {
    # Cardiovascular
    'HTA':    'hipertension arterial',
    'IAM':    'infarto agudo miocardio',
    'ACV':    'accidente cerebrovascular',
    'FA':     'fibrilacion auricular',
    'IC':     'insuficiencia cardiaca',
    'ICC':    'insuficiencia cardiaca congestiva',
    'TEP':    'tromboembolismo pulmonar',
    'TVP':    'trombosis venosa profunda',
    'SCA':    'sindrome coronario agudo',
    # Renal
    'IRC':    'insuficiencia renal cronica',
    'IRA':    'insuficiencia renal aguda',
    'RAO':    'retencion aguda orina',
    # Pulmonar
    'EPOC':   'enfermedad pulmonar obstructiva cronica',
    'SAHS':   'sindrome apnea hipopnea sueno',
    'SAOS':   'sindrome apnea obstructiva sueno',
    # Oncologia
    'CA':     'cancer',
    'NEO':    'neoplasia',
    'LMA':    'leucemia mieloide aguda',
    'LLC':    'leucemia linfocitica cronica',
    'LNH':    'linfoma no hodgkin',
    # Digestivo
    'GEA':    'gastroenteritis aguda',
    'ERGE':   'enfermedad reflujo gastroesofagico',
    'HDA':    'hemorragia digestiva alta',
    'HDB':    'hemorragia digestiva baja',
    # Hepatitis
    'VHB':    'virus hepatitis B',
    'VHC':    'virus hepatitis C',
    'VHA':    'virus hepatitis A',
    # Neurologia
    'TCE':    'traumatismo craneoencefalico',
    'TIA':    'accidente isquemico transitorio',
    'AIT':    'accidente isquemico transitorio',
    # Metabolismo
    'DM':     'diabetes mellitus',
    'DM2':    'diabetes mellitus tipo 2',
    'DM1':    'diabetes mellitus tipo 1',
    'DMG':    'diabetes mellitus gestacional',
    'HLD':    'hiperlipidemia',
    'DLP':    'dislipemia',
    # Obstetricia
    'IQ':     'intervencion quirurgica',
    'RN':     'recien nacido',
    'RNPT':   'recien nacido prematuro',
    'RPM':    'rotura prematura membranas',
    # Infeccioso
    'ITU':    'infeccion tracto urinario',
    'PCR':    'parada cardiorrespiratoria',
    'VIH':    'virus inmunodeficiencia humana',
    # Procedimientos
    'CX':     'cirugia',
    'RX':     'radiografia',
    'TAC':    'tomografia axial computarizada',
    'RM':     'resonancia magnetica',
    # Misc
    'SF':     'sindrome febril',
    'SD':     'sindrome',
    'ANTI-D': 'inmunoglobulina anti D',
}

# Catalan -> Spanish medical terms
CATALAN_TO_SPANISH = {
    'migranya':      'migranya',
    'cos estrany':   'cuerpo extranos',
    'cos':           'cuerpo',
    'estrany':       'extranos',
    'part':          'parto',
    'malaltia':      'enfermedad',
    'infeccio':      'infeccion',
    'embaras':       'embarazo',
    'diabetis':      'diabetes',
    'hipertensio':   'hipertension',
    'pneumonia':     'neumonia',
    'limfoma':       'linfoma',
    'cataracta':     'catarata',
    'artrosi':       'artrosis',
    'osteoporosi':   'osteoporosis',
    'calcul':        'calculo',
    'dret':          'derecho',
    'dreta':         'derecha',
    'esquerra':      'izquierda',
    'agut':          'agudo',
    'aguda':         'aguda',
    'cronic':        'cronico',
    'cronica':       'cronica',
    'amniodrenatge': 'amniodrenaje',
    'laceracio':     'laceracion',
    'laceracions':   'laceraciones',
    'postpart':      'postparto',
    'cesaria':       'cesarea',
}

print(f'Abbreviation dictionary : {len(ABBREVIATIONS)} entries')
print(f'Catalan dictionary      : {len(CATALAN_TO_SPANISH)} entries')

Abbreviation dictionary : 50 entries
Catalan dictionary      : 28 entries


## 3. Preprocessing Pipeline

In [5]:
ICD10_FULL   = re.compile(r'^[A-Z]\\d{2}\\.?\\d*[A-Z0-9]*$')
ICD10_INLINE = re.compile(r'\\b([A-Z]\\d{2}\\.?\\d*[A-Z0-9]*)\\b')


def remove_accents(text):
    return unicodedata.normalize('NFC',
        ''.join(c for c in unicodedata.normalize('NFD', text)
                if unicodedata.category(c) != 'Mn'))


def apply_catalan_dict(text):
    for cat, esp in sorted(CATALAN_TO_SPANISH.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'\\b' + re.escape(cat) + r'\\b', esp, text, flags=re.IGNORECASE)
    return text


def apply_abbreviations(text):
    return ' '.join(ABBREVIATIONS.get(tok.upper(), tok) for tok in text.split())


def preprocess(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.strip().lower()
    text = remove_accents(text)
    text = apply_catalan_dict(text)
    text = apply_abbreviations(text)
    return re.sub(r'\\s+', ' ', text).strip()


def extract_icd_from_literal(literal):
    if not isinstance(literal, str):
        return None
    text = literal.strip().upper()
    if ICD10_FULL.match(text):
        return text.replace('.', '')
    m = ICD10_INLINE.search(text)
    if m:
        return m.group(1).replace('.', '')
    return None


examples = ['HTA irc 6', 'MIGRANYA parto', 'cos estrany', 'VHC', 'TCE', 'J45.30', 'broncoespastica']
print(f'{"Raw":<30}  {"Preprocessed":<45}')
print('-' * 78)
for e in examples:
    print(f'{e:<30}  {preprocess(e):<45}')

Raw                             Preprocessed                                 
------------------------------------------------------------------------------
HTA irc 6                       hipertension arterial insuficiencia renal cronica 6
MIGRANYA parto                  migranya parto                               
cos estrany                     cos estrany                                  
VHC                             virus hepatitis C                            
TCE                             traumatismo craneoencefalico                 
J45.30                          j45.30                                       
broncoespastica                 broncoespastica                              


In [6]:
train_df['processed']    = train_df['Literal'].apply(preprocess)
test_df['processed']     = test_df['Literal'].apply(preprocess)
icd_df['processed']      = icd_df['Description'].apply(preprocess)
train_df['icd_shortcut'] = train_df['Literal'].apply(extract_icd_from_literal)
test_df['icd_shortcut']  = test_df['Literal'].apply(extract_icd_from_literal)

print('ICD shortcut hits in test set:')
print(test_df[test_df['icd_shortcut'].notna()][['Literal', 'icd_shortcut']])
print()
train_df[['Literal', 'processed', 'Code']].head(10)

ICD shortcut hits in test set:
Empty DataFrame
Columns: [Literal, icd_shortcut]
Index: []



,Literal,processed,Code
0,Hiperreactividad bronquial,hiperreactividad bronquial,J9809
1,broncoespástica,broncoespastica,J9801
2,miocardiopatía dilatada,miocardiopatia dilatada,I420
3,HTA irc 6,hipertension arterial insuficiencia renal cron...,Y831
4,Crisis febriles atípicas,crisis febriles atipicas,R5600
5,prótesis mama,protesis mama,Z9882
6,hernia discal parto,hernia discal parto,M5126
7,shock septico,shock septico,N189
8,puerpera,puerpera,Z390
9,podalica,podalica,O322XX0


In [7]:
icd_df.head()

,Code,D_P,Description,processed
0,A00,D,Cólera,colera
1,A000,D,"Cólera debido a Vibrio cholerae 01, biotipo ch...","colera debido a vibrio cholerae 01, biotipo ch..."
2,A001,D,"Cólera debido a Vibrio cholerae 01, biotipo El...","colera debido a vibrio cholerae 01, biotipo el..."
3,A009,D,"Cólera, no especificado","colera, no especificado"
4,A01,D,Fiebres tifoidea y paratifoidea,fiebres tifoidea y paratifoidea


In [8]:
test_df[['processed', 'icd_shortcut']].head(10)

,processed,icd_shortcut
0,amniodrenaje,None
1,hiperparatiroidismo primario,None
2,migranya parto,None
3,virus hepatitis C,None
4,absceso mama izq,None
5,miomas parto,None
6,cos estrany,None
7,traumatismo craneoencefalico,None
8,osteogenesis imperfecta,None
9,soplo sistolico,None


In [9]:
# Save preprocessed data for later use
processed_df = train_df[['processed', 'Code']]
processed_df.to_csv('data/preprocessed/preprocessed_data.csv', index=False)

# Save ICD dictionary with preprocessed descriptions
icd_df = icd_df[['processed', 'Code', 'D_P']]
icd_df.to_csv('data/preprocessed/icd_preprocessed_descriptions.csv', index=False)

# Save test set with preprocessed literals and ICD shortcuts
test_df = test_df[['processed', 'icd_shortcut']]
test_df.to_csv('data/preprocessed/test_preprocessed.csv', index=False)

## 4. Data Augmentation with ICD Descriptions

Each official ICD description is appended as a synthetic training example for its code. This dramatically helps rare codes that have few real examples.

In [10]:
X_real = train_df['processed'].tolist()
y_real = train_df['Code'].astype(str).tolist()

train_codes = set(y_real)
icd_matched = icd_df[icd_df['Code'].astype(str).isin(train_codes)]

X_synth = icd_matched['processed'].tolist()
y_synth = icd_matched['Code'].astype(str).tolist()

X_aug = X_real + X_synth
y_aug = y_real + y_synth

print(f'Real training examples   : {len(X_real):,}')
print(f'Synthetic ICD examples   : {len(X_synth):,}')
print(f'Total augmented training : {len(X_aug):,}')
print(f'Unique classes           : {len(set(y_aug)):,}')

Real training examples   : 13,700
Synthetic ICD examples   : 2,618
Total augmented training : 16,318
Unique classes           : 4,059
